# Práctica 1 · Paso 3 — Tendencias y señal de ENSO

**Cambio Climático · Ingeniería Ambiental · ESPAM MFL** · Material CC-IAM-P1-03 · Versión 1.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_03_tendencias_enso.ipynb)

**Objetivo.** Responder con los datos de su zona la pregunta central de la Unidad I: ¿hay una **evidencia** de cambio climático en el clima observado, distinguible de la **variabilidad natural**, en particular de El Niño y La Niña?

**Producto.** Series anuales 1981–2024 con su tendencia (pendiente de Sen y Mann-Kendall corregida por autocorrelación), prueba de sensibilidad sin los años de El Niño, tendencia según el año de inicio, compuestos mensuales por fase de ENSO, relación señal/ruido y estado actual de ENSO para la bitácora. Es la pieza que completa el capítulo 1 (paso 3, evaluación formativa).

**Requisito.** Haber ejecutado los cuadernos comunes `CR01_descarga_reanalisis` y `CR02_series_zona` con la misma zona. Conexión a internet para traer los índices de ENSO de NOAA (queda una copia en su Drive).

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Elija sus datos en el formulario de la celda **1** (la misma zona que en los cuadernos anteriores) y ejecute las demás en orden.

<details><summary>Secuencia de la práctica</summary>


| Orden | Cuaderno | Qué hace |
|---|---|---|
| C1 | [`CR01_descarga_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR01_descarga_reanalisis.ipynb) | Descarga ERA5 y CORe de su zona |
| C2 | [`CR02_series_zona`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR02_series_zona.ipynb) | Series diaria y mensual de la zona |
| 1 | [`IA1_01_climatologia`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_01_climatologia.ipynb) | Paso 1: climatología 1995–2014 |
| 2 | [`IA1_02_estacion_vs_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_02_estacion_vs_reanalisis.ipynb) | Paso 2: contraste con una estación |
| 3 | [`IA1_03_tendencias_enso`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_03_tendencias_enso.ipynb) | **Paso 3:** tendencias y señal de ENSO (este cuaderno) |

</details>

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del estudiante

In [ ]:
# @title Datos del estudiante y zona de trabajo (la misma en todos los cuadernos)
ESTUDIANTE = "Nombre Apellido"  # @param {type:"string"}
NIVEL = "provincia"  # @param ["provincia", "cantón"]
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
CANTON = "Manabí · Portoviejo"  # @param ["Azuay · Camilo Ponce Enriquez", "Azuay · Chordeleg", "Azuay · Cuenca", "Azuay · El Pan", "Azuay · Giron", "Azuay · Guachapala", "Azuay · Gualaceo", "Azuay · Nabon", "Azuay · Oña", "Azuay · Paute", "Azuay · Pucara", "Azuay · San Fernando", "Azuay · Santa Isabel", "Azuay · Sevilla De Oro", "Azuay · Sigsig", "Bolívar · Caluma", "Bolívar · Chillanes", "Bolívar · Chimbo", "Bolívar · Echeandia", "Bolívar · Guaranda", "Bolívar · Las Naves", "Bolívar · San Miguel", "Cañar · Azogues", "Cañar · Biblian", "Cañar · Cañar", "Cañar · Deleg", "Cañar · El Tambo", "Cañar · La Troncal", "Cañar · Suscal", "Carchi · Bolivar", "Carchi · Espejo", "Carchi · Mira", "Carchi · Montufar", "Carchi · San Pedro De Huaca", "Carchi · Tulcan", "Chimborazo · Alausi", "Chimborazo · Chambo", "Chimborazo · Chunchi", "Chimborazo · Colta", "Chimborazo · Cumanda", "Chimborazo · Guamote", "Chimborazo · Guano", "Chimborazo · Pallatanga", "Chimborazo · Penipe", "Chimborazo · Riobamba", "Cotopaxi · La Mana", "Cotopaxi · Latacunga", "Cotopaxi · Pangua", "Cotopaxi · Pujili", "Cotopaxi · Salcedo", "Cotopaxi · Saquisili", "Cotopaxi · Sigchos", "El Oro · Arenillas", "El Oro · Atahualpa", "El Oro · Balsas", "El Oro · Chilla", "El Oro · El Guabo", "El Oro · Huaquillas", "El Oro · Las Lajas", "El Oro · Machala", "El Oro · Marcabeli", "El Oro · Pasaje", "El Oro · Piñas", "El Oro · Portovelo", "El Oro · Santa Rosa", "El Oro · Zaruma", "Esmeraldas · Atacames", "Esmeraldas · Eloy Alfaro", "Esmeraldas · Esmeraldas", "Esmeraldas · La Concordia", "Esmeraldas · Muisne", "Esmeraldas · Quininde", "Esmeraldas · Rioverde", "Esmeraldas · San Lorenzo", "Galápagos · Isabela", "Galápagos · San Cristobal", "Galápagos · Santa Cruz", "Guayas · Alfredo Baquerizo Moreno", "Guayas · Balao", "Guayas · Balzar", "Guayas · Colimes", "Guayas · Crnel. Marcelino Maridueña", "Guayas · Daule", "Guayas · Duran", "Guayas · El Triunfo", "Guayas · Empalme", "Guayas · Gnral. Antonio Elizalde", "Guayas · Guayaquil", "Guayas · Isidro Ayora", "Guayas · Lomas De Sargentillo", "Guayas · Milagro", "Guayas · Naranjal", "Guayas · Naranjito", "Guayas · Nobol", "Guayas · Palestina", "Guayas · Pedro Carbo", "Guayas · Playas", "Guayas · Salitre", "Guayas · Samborondon", "Guayas · San Jacinto De Yaguachi", "Guayas · Santa Lucia", "Guayas · Simon Bolivar", "Imbabura · Antonio Ante", "Imbabura · Cotacachi", "Imbabura · Ibarra", "Imbabura · Otavalo", "Imbabura · Pimampiro", "Imbabura · San Miguel De Urcuqui", "Loja · Calvas", "Loja · Catamayo", "Loja · Celica", "Loja · Chaguarpamba", "Loja · Espindola", "Loja · Gonzanama", "Loja · Loja", "Loja · Macara", "Loja · Olmedo", "Loja · Paltas", "Loja · Pindal", "Loja · Puyango", "Loja · Quilanga", "Loja · Saraguro", "Loja · Sozoranga", "Loja · Zapotillo", "Los Ríos · Baba", "Los Ríos · Babahoyo", "Los Ríos · Buena Fe", "Los Ríos · Mocache", "Los Ríos · Montalvo", "Los Ríos · Palenque", "Los Ríos · Puebloviejo", "Los Ríos · Quevedo", "Los Ríos · Quinsaloma", "Los Ríos · Urdaneta", "Los Ríos · Valencia", "Los Ríos · Ventanas", "Los Ríos · Vinces", "Manabí · 24 De Mayo", "Manabí · Bolivar", "Manabí · Chone", "Manabí · El Carmen", "Manabí · Flavio Alfaro", "Manabí · Jama", "Manabí · Jaramijo", "Manabí · Jipijapa", "Manabí · Junin", "Manabí · Manta", "Manabí · Montecristi", "Manabí · Olmedo", "Manabí · Pajan", "Manabí · Pedernales", "Manabí · Pichincha", "Manabí · Portoviejo", "Manabí · Puerto Lopez", "Manabí · Rocafuerte", "Manabí · San Vicente", "Manabí · Santa Ana", "Manabí · Sucre", "Manabí · Tosagua", "Morona Santiago · Gualaquiza", "Morona Santiago · Huamboya", "Morona Santiago · Limon Indanza", "Morona Santiago · Logroðo", "Morona Santiago · Morona", "Morona Santiago · Pablo Sexto", "Morona Santiago · Palora", "Morona Santiago · San Juan Bosco", "Morona Santiago · Santiago", "Morona Santiago · Sucua", "Morona Santiago · Taisha", "Morona Santiago · Tiwintza", "Napo · Archidona", "Napo · Carlos Julio Arosemena Tola", "Napo · El Chaco", "Napo · Quijos", "Napo · Tena", "Orellana · Aguarico", "Orellana · La Joya De Los Sachas", "Orellana · Loreto", "Orellana · Orellana", "Pastaza · Arajuno", "Pastaza · Mera", "Pastaza · Pastaza", "Pastaza · Santa Clara", "Pichincha · Cayambe", "Pichincha · Mejia", "Pichincha · Pedro Moncayo", "Pichincha · Pedro Vicente Maldonado", "Pichincha · Puerto Quito", "Pichincha · Quito", "Pichincha · Rumiðahui", "Pichincha · San Miguel De Los Bancos", "Santa Elena · La Libertad", "Santa Elena · Salinas", "Santa Elena · Santa Elena", "Santo Domingo de los Tsáchilas · Santo Domingo", "Sucumbíos · Cascales", "Sucumbíos · Cuyabeno", "Sucumbíos · Gonzalo Pizarro", "Sucumbíos · Lago Agrio", "Sucumbíos · Putumayo", "Sucumbíos · Shushufindi", "Sucumbíos · Sucumbios", "Tungurahua · Ambato", "Tungurahua · Baños De Agua Santa", "Tungurahua · Cevallos", "Tungurahua · Mocha", "Tungurahua · Patate", "Tungurahua · Quero", "Tungurahua · San Pedro De Pelileo", "Tungurahua · Santiago De Pillaro", "Tungurahua · Tisaleo", "Zamora Chinchipe · Centinela Del Condor", "Zamora Chinchipe · Chinchipe", "Zamora Chinchipe · El Pangui", "Zamora Chinchipe · Nangaritza", "Zamora Chinchipe · Palanda", "Zamora Chinchipe · Paquisha", "Zamora Chinchipe · Yacuambi", "Zamora Chinchipe · Yantzaza", "Zamora Chinchipe · Zamora"]
# NIVEL = "provincia": se usa PROVINCIA y NO se usa CANTON.
# NIVEL = "cantón": se usa CANTON (la lista está ordenada por provincia) y NO se usa PROVINCIA:
#                   la provincia sale del propio cantón.
if NIVEL == "cantón":
    _provincia_formulario = PROVINCIA
    PROVINCIA, CANTON = CANTON.split(" · ")
    print(f"Zona elegida: cantón {CANTON}, provincia de {PROVINCIA}.")
    if _provincia_formulario != PROVINCIA:
        print(f"  Aviso: el campo PROVINCIA dice {_provincia_formulario}, pero no se usa en el nivel cantón.")
else:
    print(f"Zona elegida: provincia de {PROVINCIA}. El campo CANTON no se usa en el nivel provincia.")
    CANTON = None

REF_INI, REF_FIN = 1995, 2014     # periodo de referencia del protocolo común (no cambiar)
ANOM_INI, ANOM_FIN = 1991, 2020   # base de las anomalías mensuales para ENSO (la misma del ICEN y del ONI)
INDICE_ENSO = "ICEN"  # @param ["ICEN", "ONI"]
# ICEN: Niño 1+2, frente a la costa (recomendado en la Costa). ONI: Niño 3.4, Pacífico central.
TEMPORADA = "dic–may"  # @param ["dic–may", "ene–abr", "oct–may", "anual"]
# Temporada lluviosa con la que se clasifica cada año como El Niño, La Niña o neutro.

## 2. Entorno de trabajo

In [ ]:
import os, sys, pathlib, subprocess, datetime, platform, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, variables as V
from herramientas.reanalisis import era5, core, release
from herramientas.estaciones import calidad

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")

import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, geopandas as gpd
Z = limites.zona(PROVINCIA, CANTON)
GEOM, S = Z["geom"], Z["slug"]
DATOS = BASE / "CC_datos" / S            # datos y series comunes a las dos carreras (CR01, CR02)
RES = BASE / "CC_Ambiental" / "practica-1" / S                        # productos de este cuaderno
for c in (DATOS, RES): c.mkdir(parents=True, exist_ok=True)
print(f"Zona: {Z['nombre']} ({Z['tipo']}, DPA {Z['codigo']}, región {Z['region']})")
print(f"Datos en {DATOS} · resultados en {RES}")

from herramientas.indices import enso, extremos as X
from scipy import stats
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)

## 3. Series anuales

Cada año se resume con la **media** de las variables de estado (temperaturas, humedad, presión, viento) y la **suma** de la precipitación. Solo se usan años con los doce meses: un año incompleto sesgaría el promedio hacia los meses que sí tiene.

In [ ]:
M = pd.read_csv(DATOS / f"serie_mensual_{S}.csv", index_col="fecha", parse_dates=True)
M.index = M.index.to_period("M").to_timestamp()
FUENTES = [f for f in ("ERA5", "CORe") if f"{f}_tmed" in M]
VARS = [v for v in ("tmed", "tmax", "tmin", "pr", "hr", "q", "ps", "ws") if f"{FUENTES[0]}_{v}" in M]
COL = {"ERA5": "#C2531F", "CORe": "#4C78A8"}
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
completos = M.notna().groupby(M.index.year).sum()
ANUAL = {}
for f in FUENTES:
    d = {}
    for v in VARS:
        c = f"{f}_{v}"
        if c not in M: continue
        g = M[c].groupby(M.index.year)
        s = g.sum(min_count=12) if v == "pr" else g.mean()
        d[v] = s.where(completos[c] == 12)
    ANUAL[f] = pd.DataFrame(d)
ANUAL = pd.concat(ANUAL, axis=1)
ANUAL.index.name = "anio"
ANIO_INI, ANIO_FIN = int(ANUAL.dropna(how="all").index.min()), int(ANUAL.dropna(how="all").index.max())
print(f"{Z['nombre']}: años completos {ANIO_INI}–{ANIO_FIN} ({ANUAL.dropna(how='all').shape[0]} años) · fuentes {FUENTES}")
ANUAL.round(2).tail()

## 4. Anomalías y tendencia

La **anomalía** de cada año es su diferencia con la media del periodo de referencia 1995–2014 (en la lluvia, en porcentaje). La **pendiente de Sen** es la mediana de las pendientes entre todos los pares de años: no la arrastra un año extremo como 1983 o 1998. La **prueba de Mann-Kendall** dice si la tendencia es distinguible del azar.

**Por qué se corrige la prueba.** Los años climáticos no son independientes: un año húmedo tiende a seguir a otro. Esa persistencia hace que la prueba clásica declare significativas tendencias que no lo son. Aquí se reporta el valor p **corregido** por la autocorrelación de orden 1 (r1), con un tamaño de muestra efectivo menor, y al lado el clásico para que vea cuánto cambia la conclusión.

In [ ]:
REFM = ANUAL.loc[REF_INI:REF_FIN].mean()
ANOM_A = ANUAL - REFM
for f in FUENTES:
    ANOM_A[(f, "pr_%")] = 100 * ANOM_A[(f, "pr")] / REFM[(f, "pr")]
TEND = pd.DataFrame({(f, v): X.tendencia(ANUAL[(f, v)]) for f in FUENTES for v in VARS if (f, v) in ANUAL}).T
TEND.index.names = ["fuente", "variable"]
TEND["unidad_por_decada"] = [("mm" if v == "pr" else V.UNIDADES[v]) + "/década" for _, v in TEND.index]
display(TEND[["n", "sen_por_decada", "unidad_por_decada", "r1", "p_sin_corregir", "p_valor", "significativa_5%"]])

def panel(a, v, etiqueta, pct=False):
    for f in FUENTES:
        y = ANOM_A[(f, "pr_%" if pct else v)].dropna()
        if f == FUENTES[0]:
            a.bar(y.index, y.values, color=np.where(y.values > 0, "#C2531F" if v != "pr" else "#2C7FB8", "#4C78A8" if v != "pr" else "#B8860B"), alpha=.55, label=f"{f}: anomalía")
        else:
            a.plot(y.index, y.values, "o", ms=3, color=COL[f], label=f"{f}: anomalía")
        m, b = stats.theilslopes(y.values, y.index.values)[:2]
        t = TEND.loc[(f, v)]
        u = "%" if pct else V.UNIDADES[v]
        a.plot(y.index, b + m * y.index.values, "--", color=COL[f], lw=1.6,
               label=f"{f}: {10 * m:+.2f} {u}/década, p={t['p_valor']:.3f}")
    a.axhline(0, color="k", lw=.5); a.set_title(etiqueta, fontsize=10); a.legend(fontsize=7)

fig, ax = plt.subplots(2, 2, figsize=(15, 8))
panel(ax[0, 0], "tmed", "Temperatura media: anomalía (°C)")
panel(ax[0, 1], "pr", "Precipitación anual: anomalía (%)", pct=True)
if "tmax" in VARS: panel(ax[1, 0], "tmax", "Temperatura máxima: anomalía (°C)")
if "tmin" in VARS: panel(ax[1, 1], "tmin", "Temperatura mínima: anomalía (°C)")
fig.suptitle(f"{Z['nombre']}: anomalías respecto de {REF_INI}–{REF_FIN} y tendencia de Sen ({ANIO_INI}–{ANIO_FIN})")
fig.text(0.01, 0.005, f"Fuentes: ERA5 (Hersbach et al., 2020); CORe (NOAA/NCEP). Elaboración: {ESTUDIANTE}.", fontsize=7)
fig.tight_layout(); fig.savefig(RES / f"IA1_03_tendencias_{S}.png", dpi=300); plt.show()

## 5. ¿La tendencia depende de El Niño?

Un evento muy fuerte cerca de un extremo de la serie puede crear o borrar una tendencia. Dos pruebas:

1. **Sin los años de El Niño.** Cada año se clasifica por el índice de ENSO medio de su temporada lluviosa; se quitan los años cálidos y se recalcula la tendencia. Si la tendencia se mantiene, no la fabricó El Niño.
2. **Según el año de inicio.** La tendencia se recalcula empezando en 1981, 1985, 1990… hasta 2000, siempre hasta el último año. Una tendencia robusta conserva el signo; una frágil cambia de signo según dónde empiece.

In [ ]:
MESES_T = {"dic–may": [12, 1, 2, 3, 4, 5], "ene–abr": [1, 2, 3, 4], "oct–may": [10, 11, 12, 1, 2, 3, 4, 5],
           "anual": list(range(1, 13))}[TEMPORADA]
ONI = enso.oni(DATOS); ICEN = enso.icen(DATOS)
IDX = {"ONI": ONI, "ICEN": ICEN}
idx_t = enso.temporada(IDX[INDICE_ENSO], MESES_T, "mean")
u_c = enso.UMBRALES_ICEN["cálida débil"] if INDICE_ENSO == "ICEN" else enso.UMBRAL_ONI
u_f = enso.UMBRALES_ICEN["fría débil"] if INDICE_ENSO == "ICEN" else -enso.UMBRAL_ONI
FASE_ANIO = idx_t.map(lambda v: "El Niño" if v >= u_c else ("La Niña" if v < u_f else "Neutro")).rename("fase")
ANIOS_NINO = [int(a) for a in FASE_ANIO[FASE_ANIO == "El Niño"].index if ANIO_INI <= a <= ANIO_FIN]
print(f"Años de El Niño en la temporada {TEMPORADA} según {INDICE_ENSO}: {ANIOS_NINO}")

SENS = []
for f in FUENTES:
    for v in [w for w in ("tmed", "tmax", "tmin", "pr") if w in VARS]:
        s = ANUAL[(f, v)].dropna()
        t_all, t_sin = X.tendencia(s), X.tendencia(s.drop([a for a in ANIOS_NINO if a in s.index]))
        SENS.append({"fuente": f, "variable": v, "Sen todos": t_all["sen_por_decada"], "p todos": t_all["p_valor"],
                     "Sen sin El Niño": t_sin["sen_por_decada"], "p sin El Niño": t_sin["p_valor"], "años quitados": len(s) - t_sin["n"],
                     "mismo signo": bool(np.sign(t_all["sen_por_decada"]) == np.sign(t_sin["sen_por_decada"]))})
SENS = pd.DataFrame(SENS).set_index(["fuente", "variable"])
display(SENS)

INICIOS = sorted({ANIO_INI, *[a for a in (1985, 1990, 1995, 2000) if a > ANIO_INI]})
VENT = pd.DataFrame({(f, v): {a: X.tendencia(ANUAL[(f, v)].loc[a:].dropna())["sen_por_decada"] for a in INICIOS}
                     for f in FUENTES for v in ("tmed", "pr") if (f, v) in ANUAL})
VENT.index.name = "año de inicio"
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for a, v, u in zip(ax, ("tmed", "pr"), ("°C/década", "mm/década")):
    for f in FUENTES:
        if (f, v) in VENT: a.plot(VENT.index, VENT[(f, v)], "o-", color=COL[f], label=f)
    a.axhline(0, color="k", lw=.5); a.set_xlabel(f"Año de inicio (hasta {ANIO_FIN})"); a.set_ylabel(u)
    a.set_title(f"Pendiente de Sen de {V.NOMBRES[v].lower()} según el año de inicio", fontsize=10); a.legend(fontsize=8)
fig.tight_layout(); fig.savefig(RES / f"IA1_03_sensibilidad_inicio_{S}.png", dpi=300); plt.show()
display(VENT.round(3))

## 6. Compuestos por fase de ENSO

Cada mes se clasifica como El Niño, La Niña o neutro con la regla de persistencia del índice elegido, y se promedian las **anomalías mensuales** (base 1991–2020) de cada fase. Así se ve en qué meses y cuánto modifica ENSO el clima de su zona. Un círculo negro marca los meses en que la diferencia entre El Niño y neutro es significativa (Mann-Whitney, 5 %).

Este es el contraste clave del capítulo: **la variabilidad natural (ENSO) cambia el clima de un año a otro; la tendencia es el cambio que persiste por debajo de esa variabilidad.**

In [ ]:
from scipy.stats import mannwhitneyu
FASE_M = enso.fases(IDX[INDICE_ENSO], INDICE_ENSO).reindex(M.index)
BASE_M = M.loc[f"{ANOM_INI}":f"{ANOM_FIN}"]
CLIM = BASE_M.groupby(BASE_M.index.month).mean()
ANOM = M - CLIM.reindex(M.index.month).set_axis(M.index)
for f in FUENTES:
    ANOM[f"{f}_pr_%"] = 100 * ANOM[f"{f}_pr"] / CLIM[f"{f}_pr"].reindex(M.index.month).values
ANOM["fase"] = FASE_M
FASES = ["El Niño", "Neutro", "La Niña"]; CF = {"El Niño": "#C2531F", "Neutro": "0.5", "La Niña": "#4C78A8"}
x = np.arange(12); COMP = {}
fig, ax = plt.subplots(len(FUENTES), 2, figsize=(14, 3.8 * len(FUENTES)), squeeze=False)
for i, f in enumerate(FUENTES):
    for j, (v, et) in enumerate(((f"{f}_pr_%", "Anomalía de precipitación (%)"), (f"{f}_tmed", "Anomalía de T media (°C)"))):
        tabla = ANOM.pivot_table(index=ANOM.index.month, columns="fase", values=v, aggfunc="mean").reindex(index=range(1, 13), columns=FASES)
        COMP[v] = tabla; a = ax[i, j]
        for fa in FASES: a.plot(x, tabla[fa], "o-", color=CF[fa], label=fa)
        for m in range(1, 13):
            sel = ANOM[ANOM.index.month == m]
            a_, b_ = sel.loc[sel.fase == "El Niño", v].dropna(), sel.loc[sel.fase == "Neutro", v].dropna()
            if len(a_) >= 3 and len(b_) >= 3 and mannwhitneyu(a_, b_).pvalue < 0.05:
                a.plot(m - 1, tabla.loc[m, "El Niño"], "o", ms=12, mfc="none", mec="k")
        a.axhline(0, color="k", lw=.5); a.set_xticks(x, MESES, fontsize=8); a.set_title(f"{f}: {et}", fontsize=10); a.legend(fontsize=7)
fig.suptitle(f"{Z['nombre']}: compuestos por fase de ENSO ({INDICE_ENSO}), {ANIO_INI}–{ANIO_FIN}"); fig.tight_layout()
fig.savefig(RES / f"IA1_03_compuestos_enso_{S}.png", dpi=300); plt.show()
print(ANOM["fase"].value_counts().to_string())
display(pd.concat({k: t.round(1) for k, t in COMP.items()}, axis=1).set_axis(MESES))

## 7. Señal frente a ruido

La **señal** es el cambio acumulado por la tendencia desde el inicio de la serie; el **ruido**, la desviación estándar de los años alrededor de esa tendencia (la variabilidad natural, ENSO incluido). Cuando la señal supera una vez el ruido (S/N > 1) el cambio empieza a notarse; con S/N > 2 ya es difícil atribuirlo al azar de un año. Es una versión simple del **tiempo de emergencia** que usa el IPCC (AR6, GTI, capítulos 1 y 10).

Para ponerlo en contexto: según el IPCC (AR6, GTI, Resumen para responsables de políticas, A.1.2), la temperatura media global en superficie de 2011–2020 fue 1,09 °C [0,95 a 1,20 °C] más alta que la de 1850–1900. Compare la pendiente de su zona con el ritmo de calentamiento global que discutimos en clase.

In [ ]:
SN = []
for f in FUENTES:
    for v in [w for w in ("tmed", "tmax", "tmin", "pr") if w in VARS]:
        s = ANUAL[(f, v)].dropna()
        if len(s) < 10: continue
        m, b = stats.theilslopes(s.values, s.index.values)[:2]
        ruido = float(np.std(s.values - (b + m * s.index.values), ddof=1))
        senal = m * (s.index.max() - s.index.min())
        fila = {"fuente": f, "variable": v, "señal (cambio acumulado)": round(senal, 2), "ruido (desv. est.)": round(ruido, 2),
                "S/N": round(abs(senal) / ruido, 2) if ruido > 0 else np.nan}
        for k_ in (1, 2):
            anios = k_ * ruido / abs(m) if m != 0 else np.inf
            fila[f"año en que S/N = {k_}"] = int(round(s.index.min() + anios)) if np.isfinite(anios) and anios < 500 else "no se alcanza"
        SN.append(fila)
SN = pd.DataFrame(SN).set_index(["fuente", "variable"])
display(SN)
print("Un año posterior a", ANIO_FIN, "es una extrapolación de la tendencia lineal: no es una proyección climática (eso viene en la Unidad IV).")

## 8. El Niño en curso: datos para la bitácora

Los datos de reanálisis del curso terminan en 2024; el evento El Niño que empezó en 2026 **no entra** en el análisis climatológico. Se sigue aparte, en la **bitácora** (plantilla común en `practicas/comun/bitacora/`), con los índices más recientes, los comunicados oficiales y lo que observe en su zona, siempre marcado como dato preliminar.

In [ ]:
for k, v in IDX.items():
    v = v.dropna().tail(4)
    print(f"{k}: " + " · ".join(f"{i:%b %Y} {x:+.2f} °C" for i, x in v.items()))
print(f"Categoría ICEN del último mes: {enso.categoria_icen(float(ICEN.dropna().iloc[-1]))}")
print("Regístrelo en su bitácora con la fecha de consulta y la fuente: NOAA CPC (ONI y Niño 1+2, ERSSTv5).")
print("Contraste con el valor oficial del ICEN del IGP y con el comunicado vigente del ERFEN/INAMHI.")

## 9. Comprobaciones y guardado

In [ ]:
problemas = []
for f in FUENTES:
    t = TEND.loc[(f, "tmed")]
    if abs(t["sen_por_decada"]) > 1.0: problemas.append(f"{f}: tendencia de temperatura media de {t['sen_por_decada']} °C/década; revise la serie")
    if ANUAL[(f, "pr")].dropna().shape[0] < 30: problemas.append(f"{f}: menos de 30 años completos de precipitación")
if len(ANIOS_NINO) < 3: problemas.append("Menos de 3 temporadas de El Niño: la prueba sin El Niño y los compuestos son muy inciertos")
print("Sin problemas." if not problemas else "\n".join(problemas))
ANUAL.round(3).to_csv(RES / f"IA1_03_series_anuales_{S}.csv", encoding="utf-8")
ANOM_A.round(3).to_csv(RES / f"IA1_03_anomalias_anuales_{S}.csv", encoding="utf-8")
TEND.to_csv(RES / f"IA1_03_tendencias_{S}.csv", encoding="utf-8")
SENS.to_csv(RES / f"IA1_03_sensibilidad_el_nino_{S}.csv", encoding="utf-8")
VENT.round(4).to_csv(RES / f"IA1_03_tendencia_segun_inicio_{S}.csv", encoding="utf-8")
pd.concat({k: t for k, t in COMP.items()}, axis=1).round(2).to_csv(RES / f"IA1_03_compuestos_enso_{S}.csv", encoding="utf-8")
SN.to_csv(RES / f"IA1_03_senal_ruido_{S}.csv", encoding="utf-8")
meta = {"material": "CC-IAM-P1-03 v1.0", "estudiante": ESTUDIANTE, "zona": {k: v for k, v in Z.items() if k != "geom"},
        "periodo": [ANIO_INI, ANIO_FIN], "referencia_anomalias_anuales": [REF_INI, REF_FIN],
        "base_anomalias_mensuales": [ANOM_INI, ANOM_FIN], "indice_enso": INDICE_ENSO, "temporada": TEMPORADA,
        "anios_nino_temporada": ANIOS_NINO, "metodo_tendencia": "pendiente de Sen; Mann-Kendall con n efectivo (r1), Yue y Wang (2004)",
        "fuentes_enso": enso.URL, "fecha": datetime.date.today().isoformat()}
(RES / "IA1_03_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(sorted(p.name for p in RES.glob("IA1_03_*")))

## 10. Preguntas para el capítulo 1

1. ¿Cuánto ha cambiado la temperatura media de su zona por década? ¿Es significativa con la prueba corregida? ¿Cambió la conclusión respecto de la prueba clásica? Explique por qué con el valor de r1.
2. ¿Calientan igual las máximas y las mínimas? ¿Qué implica para el rango diario de temperatura?
3. ¿La precipitación anual tiene tendencia? Compare su magnitud con la variabilidad entre años: ¿qué pesa más en su zona, la tendencia o ENSO?
4. ¿La tendencia sobrevive sin los años de El Niño y con distintos años de inicio? ¿Qué variable tiene la tendencia más robusta y cuál la más frágil?
5. ¿En qué meses y cuánto cambian la lluvia y la temperatura en El Niño y en La Niña? ¿Son significativos?
6. Con la relación señal/ruido: ¿el calentamiento de su zona ya emergió de la variabilidad natural? Compárelo con el calentamiento global del AR6 y use el lenguaje calibrado del IPCC (probable, muy probable; confianza baja, media, alta) para formular su conclusión.
7. Registre en la bitácora los valores más recientes del ICEN y del ONI.

**Para el informe:** periodo de años completos, criterio de año completo, índice y temporada de ENSO, años quitados en la prueba de sensibilidad, método de la tendencia y su corrección.

**Siguiente:** arme el borrador del capítulo 1 con la [plantilla](capitulo-1_plantilla.md).

**Citas.** Sen, P. K. (1968). *J. Am. Stat. Assoc.*, 63, 1379–1389 · Mann, H. B. (1945). *Econometrica*, 13, 245–259 · Kendall, M. G. (1975). *Rank Correlation Methods*. Griffin · Yue, S. y Wang, C. (2004). *Water Resources Management*, 18, 201–218 · IPCC (2021). *Summary for Policymakers*. En: *Climate Change 2021: The Physical Science Basis*. Cambridge University Press · Huang, B. et al. (2017). ERSSTv5. *J. Climate*, 30, 8179–8205.